In [ ]:
import whisper
import sounddevice as sd
import numpy as np
import queue
import threading
import time
import requests
import os
from scipy.io.wavfile import write

# Load Whisper model
model = whisper.load_model("base")

# Audio settings
samplerate = 16000
block_duration = 10  # seconds per block read
record_duration = 60  # accumulate for 1 minute
channels = 2
device = 0  # BlackHole

# Audio buffer
q = queue.Queue()

def audio_callback(indata, frames, time_info, status):
    if status:
        print("⚠️", status)
    q.put(indata.copy())

def record_audio_loop():
    accumulated_audio = []

    with sd.InputStream(samplerate=samplerate, channels=channels, device=device, callback=audio_callback):
        print("🎤 Recording started. Listening to speaker audio...")

        while True:
            block = q.get()
            accumulated_audio.append(block)

            # Calculate accumulated time
            total_duration = len(accumulated_audio) * block.shape[0] / samplerate

            if total_duration >= record_duration:
                # Combine blocks
                print(f"⏳ Accumulated duration: {total_duration:.1f} sec")
                audio_data = np.concatenate(accumulated_audio, axis=0)

                # Convert stereo → mono
                if channels == 2:
                    audio_data = audio_data.mean(axis=1)

                # Normalize to int16
                audio_int16 = np.int16(audio_data * 32767)

                # Save temp file
                timestamp = int(time.time())
                filename = f"../data/AI_Meeting_Assistant/temp_{timestamp}.wav"
                os.makedirs(os.path.dirname(filename), exist_ok=True)
                write(filename, samplerate, audio_int16)

                # Transcribe with Whisper
                if os.path.exists(filename):
                    print("🔍 Transcribing...")
                    result = model.transcribe(filename)
                    transcript = result['text'].strip()
                    os.remove(filename)

                    if len(transcript.split()) >= 5:
                        print(f"\n🔊 Detected speech:\n{transcript}\n")
                        print("🧠 Summary:")
                        print(summarize_with_gemma(transcript))
                        print("\n✅ Action Items:")
                        print(extract_actions_with_gemma(transcript))
                    else:
                        print("⚠️ Transcript too short. Skipping...")

                accumulated_audio = []  # reset

def summarize_with_gemma(text):
    prompt = f"Summarize the following meeting excerpt in 2-3 bullet points:\n{text}"
    return call_ollama(prompt)

def extract_actions_with_gemma(text):
    prompt = f"Extract action items or decisions from the following conversation:\n{text}"
    return call_ollama(prompt)

def call_ollama(prompt):
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": "gemma", "prompt": prompt, "stream": False}
        )
        return response.json().get("response", "").strip()
    except Exception as e:
        return f"❌ Error calling Ollama: {e}"

# Start the assistant
record_thread = threading.Thread(target=record_audio_loop, daemon=True)
record_thread.start()

# Keep alive
while True:
    time.sleep(1)


In [ ]:
import threading
import os

stop_event = threading.Event()


def stop_recording():
    stop_event.set()
    os._exit(0)

stop_recording()


In [ ]:
import threading

print("🧵 Active threads:")
for thread in threading.enumerate():
    print(f"• Name: {thread.name}, Alive: {thread.is_alive()}, Daemon: {thread.daemon}")


In [5]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo  # Python 3.9+

# Your timestamp
timestamp = 1754040797.826284

# Convert to UTC datetime (timezone-aware)
utc_dt = datetime.fromtimestamp(timestamp, tz=timezone.utc)
print("UTC:", utc_dt)

# Convert to Central Time (automatically handles CDT vs CST)
cdt_dt = utc_dt.astimezone(ZoneInfo("America/Chicago"))
print("CDT:", cdt_dt)


UTC: 2025-08-01 09:33:17.826284+00:00
CDT: 2025-08-01 04:33:17.826284-05:00
